## Install Dependencies

In [1]:
!apt-get update
!apt-get install -y ffmpeg
!pip install -U openai-whisper jiwer
!pip install -U tqdm

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,328 kB]
Get:13 https://r2u.

### Verify Installation

In [2]:
import whisper
import torch
import subprocess

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Whisper loaded:", whisper.load_model("base") is not None)

subprocess.run(["ffmpeg", "-version"], capture_output=True)
print("FFmpeg OK")

Torch: 2.9.0+cu126
CUDA available: True


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 152MiB/s]


Whisper loaded: True
FFmpeg OK


## Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Import and Configuration

In [4]:

import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple
from dataclasses import dataclass, asdict
import whisper
from jiwer import wer as compute_wer


CONFIG = {
    "cha_dir": Path("/content/drive/MyDrive/asr/data/cha"),
    "audio_dir": Path("/content/drive/MyDrive/asr/data/songs"),
    "output_dir": Path("/content/drive/MyDrive/asr/output/whisper_children_dataset"),
    "whisper_model": "base",
    "sample_size": None,  # None = tous, ou 500 pour tester
    "train_ratio": 0.8,
    "sample_rate": 16000,
    "audio_extensions": [".wav", ".mp3", ".m4a", ".flac"],
    "quality_threshold": 0.9,  # 0.6 (loose) to 0.85 (strict)
    # Whether to compute expensive audio features
    "analyze_audio_features": True,  # Set to False to speed up analysis
    # Analysis output
    "save_quality_visualization": True,
    "save_filtered_segments": True,
}

# CHILDES roles
CHILD_ROLES = {"Target_Child", "Child", "Sibling", "Peer", "Playmate"}
ADULT_ROLES = {"Investigator", "Teacher", "Mother", "Father", "Adult", "Caregiver", "Parent"}



## Zone 1: File Matching


COMPLETE PIPELINE NOTEBOOK
Input: Dataset folders (data/ca + data/songs)
Output: Training dataset for Whisper fine-tuning (children voices only)

Flow:
1. Match .cha ↔ Audio files
2. Extract .cha segments (word-level timestamps)
3. Segment audio files based on timestamps
4. Evaluate Whisper baseline (children only)
5. Calculate WER (children only)
6. Create training dataset (JSONL + metadata)


In [5]:
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List, Union

@dataclass
class WorSegment:
    speaker: str
    text: str
    words: list  # [(word, start, end)]
    file_name: str = ""  # Ajouter le nom du fichier source


def extract_wor_segments(path: Union[Path, str], debug: bool = False) -> List[WorSegment]:
    """
    Extraire segments %wor d'un fichier .cha

    Args:
        path: Chemin vers un fichier .cha OU un dossier contenant des .cha
        debug: Afficher les infos de parsing

    Returns:
        Liste de WorSegment
    """
    path = Path(path)

    if path.is_dir():
        # Si c'est un dossier, traiter tous les .cha
        return _extract_from_directory(path, debug=debug)
    elif path.is_file():
        # Si c'est un fichier, le traiter
        return _extract_from_file(path, debug=debug)
    else:
        raise FileNotFoundError(f"Chemin invalide: {path}")


def _extract_from_directory(cha_dir: Path, debug: bool = False) -> List[WorSegment]:
    """Extraire de tous les fichiers .cha d'un dossier (récursivement)"""

    # Chercher les .cha dans le dossier ET les sous-dossiers
    cha_files = sorted(cha_dir.glob("*.cha")) + sorted(cha_dir.glob("**/*.cha"))
    # Enlever les doublons
    cha_files = sorted(set(cha_files))

    if not cha_files:
        print(f"Aucun fichier .cha trouvé dans {cha_dir}")
        return []

    if debug:
        print(f"Traitement de {len(cha_files)} fichiers .cha\n")

    all_segments = []

    for cha_file in cha_files:
        if debug:
            print(f" {cha_file.name}...", end=" ")

        segments = _extract_from_file(cha_file, debug=False)
        all_segments.extend(segments)

        if debug:
            print(f"({len(segments)} segments)")

    if debug:
        print(f"\n{'=' * 60}")
        print(f"Total: {len(all_segments)} segments de {len(cha_files)} fichiers")
        print(f"{'=' * 60}\n")

    return all_segments


In [6]:

def _extract_from_file(cha_file: Path, debug: bool = False) -> List[WorSegment]:
    """Extraire de un seul fichier .cha"""

    segments = []
    current_speaker = None
    file_name = cha_file.stem

    with cha_file.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip()

            # ── tour principal
            if line.startswith("*"):
                current_speaker = line.split(":", 1)[0].replace("*", "").strip()

            # ── word tier
            elif line.startswith("%wor:"):
                if not current_speaker:
                    continue

                # Extraire la partie après "%wor:"
                wor_content = line.split(":", 1)[1].strip()

                # Nettoyer les caractères de contrôle
                wor_content = wor_content.replace('\x15', '')

                # Parser simple : splitter par espaces et apparier word + timestamp
                tokens = wor_content.split()

                words = []
                i = 0
                while i < len(tokens):
                    token = tokens[i]

                    # Vérifier si c'est un timestamp (format XXXXX_XXXXX)
                    if re.match(r"^\d{5,}_\d{5,}$", token):
                        # C'est un timestamp → l'attacher au mot précédent
                        if words:
                            word, _, _ = words[-1]
                            match = re.match(r"(\d+)_(\d+)", token)
                            if match:
                                start, end = int(match.group(1)), int(match.group(2))
                                words[-1] = (word, start, end)
                        i += 1
                        continue

                    # Sinon, c'est un mot
                    words.append((token, None, None))
                    i += 1

                # Filtrer : garder seulement les mots avec timestamps
                words_with_ts = [(w, s, e) for w, s, e in words if s is not None and e is not None]

                if not words_with_ts:
                    continue

                # Nettoyer le texte : enlever les ponctuations isolées
                clean_words = [w for w, _, _ in words_with_ts if w not in ('?', '.', ',', '!', '+...')]

                if clean_words:
                    clean_text = " ".join(clean_words)

                    segments.append(
                        WorSegment(
                            speaker=current_speaker,
                            text=clean_text,
                            words=words_with_ts,
                            file_name=file_name
                        )
                    )

                    if debug and len(segments) <= 3:
                        print(f"\n {current_speaker}")
                        print(f"   Text: {clean_text[:70]}")
                        print(f"   Words: {words_with_ts[:3]}...")

    if debug:
        print(f"\n{'=' * 60}")
        print(f"Total segments ({file_name}): {len(segments)}")
        print(f"{'=' * 60}")

    return segments

In [7]:
def print_statistics(segments: List[WorSegment]):
    """Afficher statistiques détaillées sur les segments"""

    if not segments:
        print(" Aucun segment trouvé")
        return

    print(f"\n{'=' * 60}")
    print("STATISTIQUES")
    print(f"{'=' * 60}")

    # Par speaker
    by_speaker = {}
    by_file = {}

    for seg in segments:
        # Par speaker
        by_speaker.setdefault(seg.speaker, []).append(seg)

        # Par fichier
        by_file.setdefault(seg.file_name, []).append(seg)

    print(f"\nPar speaker ({len(by_speaker)} speakers):")
    for speaker in sorted(by_speaker.keys()):
        segs = by_speaker[speaker]
        total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segs) / 1000
        print(f"   {speaker:20} {len(segs):3d} segments | {total_duration:6.1f}s audio")

    print(f"\nPar fichier ({len(by_file)} fichiers):")
    for file_name in sorted(by_file.keys()):
        segs = by_file[file_name]
        total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segs) / 1000
        print(f"   {file_name:30} {len(segs):3d} segments | {total_duration:6.1f}s audio")

    # Stats globales
    total_duration = sum(s.words[-1][2] - s.words[0][1] for s in segments) / 1000 / 60
    avg_words = sum(len(s.words) for s in segments) / len(segments)

    print(f"\nGlobales:")
    print(f"   Total segments: {len(segments)}")
    print(f"   Total audio: {total_duration:.1f} minutes")
    print(f"   Mots par segment (moyennes): {avg_words:.1f}")
    print(f"{'=' * 60}\n")


if __name__ == "__main__":
    # Exemple 1: Traiter UN fichier
    print("=" * 60)
    print("EXEMPLE 1: UN FICHIER")
    print("=" * 60)
    segments_single = extract_wor_segments(Path("/content/drive/MyDrive/asr/data/cha/1/01-1a.cha"), debug=True)

    # Exemple 2: Traiter UN DOSSIER ENTIER
    print("\n" + "=" * 60)
    print("EXEMPLE 2: DOSSIER ENTIER")
    print("=" * 60)
    segments_all = extract_wor_segments(Path("/content/drive/MyDrive/asr/data/cha/1/"), debug=True)

    # Afficher statistiques
    print_statistics(segments_all)

    # Exemples
    if segments_all:
        print(f"{'=' * 60}")
        print("EXEMPLES DE SEGMENTS:")
        print(f"{'=' * 60}")
        for i, seg in enumerate(segments_all[:5]):
            print(f"\n[{i + 1}] {seg.speaker} ({seg.file_name})")
            print(f"  Text: {seg.text}")
            if seg.words:
                print(f"  Time: {seg.words[0][1]} → {seg.words[-1][2]} ms")
                print(f"  Words: {seg.words[:3]}...")





EXEMPLE 1: UN FICHIER

 KAT
   Text: un escargot Dylan
   Words: [('un', 10438, 10478), ('escargot', 10739, 10919), ('Dylan', 10919, 11419)]...

 KAT
   Text: comment
   Words: [('comment', 35270, 35770)]...

 WIL
   Text: moi fais la fourmi moi
   Words: [('moi', 38747, 39670), ('fais', 40011, 40612), ('la', 41896, 42136)]...

Total segments (01-1a): 26

EXEMPLE 2: DOSSIER ENTIER
Traitement de 112 fichiers .cha

 01-1a.cha... (26 segments)
 01-1b.cha... (8 segments)
 01-2.cha... (211 segments)
 01-3a.cha... (91 segments)
 01-3b.cha... (71 segments)
 01-3c.cha... (89 segments)
 01-3d.cha... (181 segments)
 01-4.cha... (82 segments)
 02-5.cha... (217 segments)
 02-7.cha... (397 segments)
 02-8.cha... (8 segments)
 02-9a.cha... (224 segments)
 02-9b.cha... (118 segments)
 02-9c.cha... (40 segments)
 02-9d.cha... (138 segments)
 02-9e.cha... (17 segments)
 02-9f.cha... (67 segments)
 03-10a.cha... (75 segments)
 03-10b.cha... (130 segments)
 03-13a.cha... (66 segments)
 03-13b.cha... (98 

In [8]:
def find_matching_files(cha_dir: Path, audio_dir: Path, extensions: List[str]) -> Dict:
    """Match .cha with audio files by relative path (respects subdirectories)"""

    cha_files = sorted(cha_dir.glob("**/*.cha"))
    audio_files = []
    for ext in extensions:
        audio_files.extend(audio_dir.glob(f"**/*{ext}"))

    # Créer dicts: relative_path_with_stem → fichier
    # Exemple: "1/01-1a" pour data/cha/1/01-1a.cha
    cha_by_path = {}
    for f in cha_files:
        relative_stem = str(f.relative_to(cha_dir).with_suffix(""))  # "1/01-1a"
        cha_by_path[relative_stem] = f

    audio_by_path = {}
    for f in audio_files:
        relative_stem = str(f.relative_to(audio_dir).with_suffix(""))
        audio_by_path[relative_stem] = f

    # Matcher: chercher les mêmes chemins relatifs
    matched = []
    cha_missing = []
    audio_orphans = []

    for relative_path in cha_by_path:
        if relative_path in audio_by_path:
            matched.append((cha_by_path[relative_path], audio_by_path[relative_path]))
        else:
            cha_missing.append(cha_by_path[relative_path])

    for relative_path in audio_by_path:
        if relative_path not in cha_by_path:
            audio_orphans.append(audio_by_path[relative_path])

    return {
        "matched": matched,
        "cha_missing_audio": cha_missing,
        "audio_orphans": audio_orphans,
        "total_cha": len(cha_files),
        "total_audio": len(audio_files),
        "matched_count": len(matched)
    }


def print_matching_report(result: Dict):
    """Afficher le rapport de matching"""
    print("\n" + "="*70)
    print("STEP 1: FILE MATCHING (by relative path)")
    print("="*70)
    print(f"\n Found:")
    print(f"   Total .cha files:       {result['total_cha']}")
    print(f"   Total audio files:      {result['total_audio']}")
    print(f"   Matched pairs:        {result['matched_count']}")
    print(f"   .cha missing audio:  {len(result['cha_missing_audio'])}")
    print(f"   Audio orphans:       {len(result['audio_orphans'])}")

    if result['cha_missing_audio']:
        print(f"\n   Missing audio for:")
        for cha in result['cha_missing_audio'][:10]:
            print(f"      - {cha.name}")
        if len(result['cha_missing_audio']) > 10:
            print(f"      ... and {len(result['cha_missing_audio']) - 10} more")

    if result['audio_orphans']:
        print(f"\n   Audio without .cha:")
        for audio in result['audio_orphans'][:10]:
            print(f"      - {audio.name}")
        if len(result['audio_orphans']) > 10:
            print(f"      ... and {len(result['audio_orphans']) - 10} more")

    print("\n" + "="*70 + "\n")


In [9]:
  # ZONE 1: Matching
print("\n" + "="*70)
print("ZONE 1: FILE MATCHING")
print("="*70)
match_result = find_matching_files(CONFIG["cha_dir"], CONFIG["audio_dir"], CONFIG["audio_extensions"])
print_matching_report(match_result)

if not match_result["matched"]:
    print("❌ No matched pairs found!")



ZONE 1: FILE MATCHING

STEP 1: FILE MATCHING (by relative path)

 Found:
   Total .cha files:       247
   Total audio files:      245
   Matched pairs:        245
   .cha missing audio:  2
   Audio orphans:       0

   Missing audio for:
      - 03-13c.cha
      - 05-23a.cha




## Zone 2: Segment Extraction

In [10]:
def extract_segments_from_matched(matched_pairs: List[Tuple[Path, Path]]) -> List[WorSegment]:
    """Extract .cha segments from matched files only"""

    print("="*70)
    print("STEP 2: EXTRACT .CHA SEGMENTS")
    print("="*70)

    all_segments = []

    for i, (cha_file, audio_file) in enumerate(matched_pairs):
        segments = extract_wor_segments(cha_file, debug=False)
        all_segments.extend(segments)

        if (i + 1) % 50 == 0:
            print(f"  ✓ {i + 1}/{len(matched_pairs)} files")

    print(f"\nExtracted {len(all_segments)} segments with timestamps\n")
    return all_segments

In [11]:
 # ZONE 2: Extract segments from .cha files
print("\n" + "="*70)
print("ZONE 2: EXTRACT SEGMENTS FROM .CHA")
print("="*70)
segments = extract_segments_from_matched(match_result["matched"])




ZONE 2: EXTRACT SEGMENTS FROM .CHA
STEP 2: EXTRACT .CHA SEGMENTS
  ✓ 50/245 files
  ✓ 100/245 files
  ✓ 150/245 files
  ✓ 200/245 files

Extracted 29272 segments with timestamps



## Zone 3: Audio Segmentation

In [12]:
"""
AUDIO SEGMENTER WITH MAX LIMIT
Permet de limiter le nombre de segments à traiter pour respecter les limites de calcul
"""
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple

class AudioSegmenter:
    """Segment audio files based on timestamps with optional limit"""

    def __init__(self, output_dir: Path, sample_rate: int = 16000):
        self.output_dir = output_dir
        self.sample_rate = sample_rate
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.speaker_dirs = {}

    def _get_speaker_dir(self, speaker: str) -> Path:
        """Get or create speaker directory"""
        if speaker not in self.speaker_dirs:
            d = self.output_dir / speaker
            d.mkdir(exist_ok=True)
            self.speaker_dirs[speaker] = d
        return self.speaker_dirs[speaker]

    def extract_segment(self, audio_file: Path, start_ms: int, end_ms: int, output_path: Path) -> bool:
        """Extract audio segment using ffmpeg"""
        if not audio_file.exists():
            return False

        start_sec = start_ms / 1000.0
        duration_sec = (end_ms - start_ms) / 1000.0

        cmd = [
            "ffmpeg", "-i", str(audio_file),
            "-ss", str(start_sec), "-t", str(duration_sec),
            "-acodec", "pcm_s16le", "-ar", str(self.sample_rate), "-ac", "1",
            "-y", str(output_path)
        ]

        try:
            subprocess.run(cmd, check=True, capture_output=True, timeout=10)
            return True
        except:
            return False

    def segment_all(self, segments: List[WorSegment], matched_pairs: List[Tuple[Path, Path]],
                   max_segments: int = None) -> Dict:
        """
        Segment audio files with optional limit

        Args:
            segments: List of WorSegment objects
            matched_pairs: List of (cha_file, audio_file) tuples
            max_segments: Maximum number of segments to process (None = all)

        Returns:
            Dict with:
            - extracted: List of successfully extracted segments
            - skipped: Number of skipped segments (no audio file)
            - stopped_at: Number processed before stopping (if max reached)
            - limited: Boolean, whether max_segments was reached
        """

        print("="*70)
        print(" STEP 3: SEGMENT AUDIO FILES")
        print("="*70)

        audio_lookup = {audio.stem: audio for _, audio in matched_pairs}

        # Déterminer le nombre de segments à traiter
        segments_to_process = segments
        limited = False

        if max_segments is not None:
            if len(segments) > max_segments:
                segments_to_process = segments[:max_segments]
                limited = True
                print(f"\n LIMIT SET: Processing {max_segments}/{len(segments)} segments")
            else:
                print(f"\n Processing all {len(segments)} segments (limit: {max_segments})")
        else:
            print(f"\n Processing all {len(segments)} segments (no limit)")

        results = []
        skipped = 0

        for i, seg in enumerate(segments_to_process):
            audio_file = audio_lookup.get(seg.file_name)
            if not audio_file or not audio_file.exists():
                skipped += 1
                continue

            segment_id = f"{seg.file_name}_{seg.speaker}_{i:05d}"
            speaker_dir = self._get_speaker_dir(seg.speaker)
            output_path = speaker_dir / f"{segment_id}.wav"

            start_ms = seg.words[0][1]
            end_ms = seg.words[-1][2]

            if self.extract_segment(audio_file, start_ms, end_ms, output_path):
                results.append({
                    "segment_id": segment_id,
                    "speaker": seg.speaker,
                    "file_name": seg.file_name,
                    "audio_path": str(output_path),
                    "duration_ms": end_ms - start_ms,
                    "text": seg.text,
                    "num_words": len(seg.words)
                })

            if (i + 1) % 200 == 0:
                print(f"  ✓ {i + 1}/{len(segments_to_process)} segments")

        print(f"\n Segmented {len(results)} audio files")
        if skipped > 0:
            print(f"   Skipped (no audio): {skipped}")

        if limited:
            print(f"   STOPPED AT LIMIT: {len(segments_to_process)} processed")
            print(f"   Remaining: {len(segments) - len(segments_to_process)} segments not processed")

        print()

        return {
            "extracted": results,
            "skipped": skipped,
            "stopped_at": len(segments_to_process),
            "limited": limited,
            "total_segments": len(segments)
        }

# ============================================================================
# HELPER: Resume segmentation
# ============================================================================

def resume_segmentation(segments: List[WorSegment], matched_pairs: List[Tuple[Path, Path]],
                       output_dir: Path, start_from: int = 0, max_segments: int = None) -> Dict:
    """
    Resume segmentation from a specific point

    Utile si tu veux reprendre après avoir atteint ta limite

    Args:
        segments: List of all WorSegment objects
        matched_pairs: List of (cha_file, audio_file) tuples
        output_dir: Output directory
        start_from: Index to start from (0 = beginning)
        max_segments: Maximum segments to process from start_from

    Returns:
        Dict with results
    """

    print("="*70)
    print("RESUME SEGMENTATION")
    print("="*70)
    print(f"\n  Resuming from segment {start_from}")
    print(f"  Total remaining: {len(segments) - start_from}\n")

    # Get remaining segments
    remaining_segments = segments[start_from:]

    segmenter = AudioSegmenter(output_dir)
    results = segmenter.segment_all(remaining_segments, matched_pairs, max_segments)

    # Adjust stopped_at to reflect actual position in original list
    results["stopped_at"] = start_from + results["stopped_at"]

    return results

In [13]:
 # ZONE 3: Segment audio files
print("\n" + "="*70)
print("ZONE 3: CREATE AUDIO SEGMENTS")
print("="*70)
segmenter = AudioSegmenter(CONFIG["output_dir"] / "audio_segments")
segment_all_result = segmenter.segment_all(segments, match_result["matched"], max_segments=1000)
audio_segments = segment_all_result["extracted"]




ZONE 3: CREATE AUDIO SEGMENTS
 STEP 3: SEGMENT AUDIO FILES

 LIMIT SET: Processing 1000/29272 segments
  ✓ 200/1000 segments
  ✓ 400/1000 segments
  ✓ 600/1000 segments
  ✓ 800/1000 segments
  ✓ 1000/1000 segments

 Segmented 1000 audio files
   STOPPED AT LIMIT: 1000 processed
   Remaining: 28272 segments not processed



## Zone 4: Extract Speakers Metadata

In [14]:
def extract_all_speakers_info(matched_pairs: List[Tuple[Path, Path]]) -> Dict[str, str]:
    """Extract speaker roles from all .cha files"""

    all_speakers = {}

    for cha_file, _ in matched_pairs:
        with cha_file.open(encoding="utf-8") as f:
            for line in f:
                if line.startswith("@Participants:"):
                    participants_str = line.split(":", 1)[1].strip()
                    for participant in participants_str.split(","):
                        participant = participant.strip()
                        parts = participant.rsplit(" ", 1)
                        if len(parts) == 2:
                            speaker_name, role = parts
                            all_speakers[speaker_name] = role

    return all_speakers

In [15]:
  # ZONE 4: Get speaker info
print("\n" + "="*70)
print("ZONE 4: EXTRACT SPEAKER ROLES")
print("="*70)
speakers_info = extract_all_speakers_info(match_result["matched"])


ZONE 4: EXTRACT SPEAKER ROLES


## ZONE 5: DATA QUALITY ANALYSIS (NEW STEP!)

In [16]:
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm # Import tqdm for progress bar

def convert_audio_segments_to_wor_segments(audio_segments):
    """
    Convert audio_segments dict format to WorSegment format for analysis.

    This is the bridge between your existing pipeline and the analysis step.
    """
    analysis_segments = []

    for audio_seg in audio_segments:
        # Extract word-level timing if available from segment metadata
        # If not available, we estimate from text and timing
        text = audio_seg["text"]
        words_list = text.split()

        # Create word list with timing
        # (In real scenario, you'd have actual timestamps from the .cha file)
        words = [(word, i*100, (i+1)*100) for i, word in enumerate(words_list)]

        wor_seg = WorSegment(
            speaker=audio_seg["speaker"],
            text=text,
            words=words,
            file_name=audio_seg["file_name"]
            # audio_path=Path(audio_seg["audio_path"]), # WorSegment does not have audio_path
            # cha_path=Path("")  # Not needed for this analysis
        )
        analysis_segments.append(wor_seg)

    return analysis_segments

    # 🆕 ADD THIS QUALITY ANALYSIS FUNCTION
# ────────────────────────────────────────────────────────────────────────

def extract_audio_segment_features(audio_segments_dict):
    """
    Extract features directly from audio_segments dict format (your current format).

    This is simpler than the full WorSegment approach because you already have
    the audio files and ground truth text.
    """
    rows = []

    print(f"\n{'='*70}")
    print("EXTRACTING FEATURES FROM AUDIO SEGMENTS")
    print(f"{'='*70}\n")

    for i, seg in enumerate(tqdm(audio_segments_dict, desc="Processing segments")):
        audio_path = Path(seg["audio_path"])

        # ── Basic metadata ──
        features = {
            'segment_id': seg["segment_id"],
            'speaker': seg["speaker"],
            'file_name': seg["file_name"],
            'duration_ms': seg["duration_ms"],
            'text': seg["text"],
            'n_words': len(seg["text"].split()),
            'text_length': len(seg["text"]),
        }

        # ── Calculate speech rate (from metadata) ──
        if seg["duration_ms"] > 0:
            features['speech_rate_wps'] = len(seg["text"].split()) / (seg["duration_ms"] / 1000)
        else:
            features['speech_rate_wps'] = 0

        # ── Load and analyze audio ──
        if audio_path.exists():
            try:
                y, sr = librosa.load(str(audio_path), sr=None)

                # Energy analysis
                rms = librosa.feature.rms(y=y)[0]
                features['energy_mean'] = np.mean(rms)
                features['energy_std'] = np.std(rms)

                # Speech activity ratio (energy-based VAD)
                threshold = np.percentile(rms, 30)
                speech_frames = rms > threshold
                features['speech_activity_ratio'] = np.sum(speech_frames) / len(speech_frames)

                # Zero-crossing rate
                zcr = librosa.feature.zero_crossing_rate(y)[0]
                features['zcr_mean'] = np.mean(zcr)

                # Dynamic range
                db = librosa.power_to_db(np.abs(librosa.stft(y))**2, ref=np.max)
                features['dynamic_range'] = np.max(db) - np.min(db)

                # Spectral features
                spectral_centroids = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
                features['spectral_centroid'] = np.mean(spectral_centroids)

            except Exception as e:
                print(f"  ⚠️  Error loading {audio_path}: {e}")
                features.update({
                    'energy_mean': np.nan,
                    'energy_std': np.nan,
                    'speech_activity_ratio': np.nan,
                    'zcr_mean': np.nan,
                    'dynamic_range': np.nan,
                    'spectral_centroid': np.nan,
                })
        else:
            features.update({
                'energy_mean': np.nan,
                'energy_std': np.nan,
                'speech_activity_ratio': np.nan,
                'zcr_mean': np.nan,
                'dynamic_range': np.nan,
                'spectral_centroid': np.nan,
            })

        rows.append(features)

    df = pd.DataFrame(rows)
    print(f"\n✓ Extracted features for {len(df)} segments\n")
    return df


# 🆕 ADD THIS QUALITY SCORING CLASS
# ────────────────────────────────────────────────────────────────────────

class AudioQualityScorer:
    """
    Score quality of audio segments based on acoustic and metadata features.
    Adjusted for your audio_segments format.
    """

    def __init__(self):
        self.rules = {
            'speech_rate': {
                'min': 0.5,
                'max': 6,
                'penalty': 0.3,
                'reason': 'Speech rate outside normal range (1-5 words/sec)'
            },
            'duration': {
                'min_ms': 5000,
                'max_ms': 60000,
                'penalty': 0.2,
                'reason': 'Segment duration outside acceptable range (1-60 sec)'
            },
            'text_length': {
                'min_chars': 10,
                'penalty': 0.1,
                'reason': 'Text too short (< 10 characters)'
            },
            'speech_activity': {
                'min_ratio': 0.3,
                'penalty': 0.25,
                'reason': 'Too much silence/background noise (<40% speech)'
            },
            'energy': {
                'min_mean': 0.01,
                'penalty': 0.1,
                'reason': 'Audio too quiet'
            },
            'dynamic_range': {
                'min_db': 5.0,
                'penalty': 0.15,
                'reason': 'Audio has poor dynamic range (too compressed)'
            }
        }

    def score(self, df):
        """Compute quality scores for each segment."""
        scores = np.ones(len(df))
        issues = {i: [] for i in range(len(df))}

        # Rule 1: Speech rate
        bad = (df['speech_rate_wps'] < self.rules['speech_rate']['min']) | \
              (df['speech_rate_wps'] > self.rules['speech_rate']['max'])
        scores[bad] -= self.rules['speech_rate']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['speech_rate']['reason'])

        # Rule 2: Duration
        bad = (df['duration_ms'] < self.rules['duration']['min_ms']) | \
              (df['duration_ms'] > self.rules['duration']['max_ms'])
        scores[bad] -= self.rules['duration']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['duration']['reason'])

        # Rule 3: Text length
        bad = df['text_length'] < self.rules['text_length']['min_chars']
        scores[bad] -= self.rules['text_length']['penalty']
        for idx in df[bad].index:
            issues[idx].append(self.rules['text_length']['reason'])

        # Rule 4: Speech activity (if available)
        if 'speech_activity_ratio' in df.columns:
            bad = (df['speech_activity_ratio'] < self.rules['speech_activity']['min_ratio']) & \
                  (df['speech_activity_ratio'].notna())
            scores[bad] -= self.rules['speech_activity']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['speech_activity']['reason'])

        # Rule 5: Energy (if available)
        if 'energy_mean' in df.columns:
            bad = (df['energy_mean'] < self.rules['energy']['min_mean']) & \
                  (df['energy_mean'].notna())
            scores[bad] -= self.rules['energy']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['energy']['reason'])

        # Rule 6: Dynamic range (if available)
        if 'dynamic_range' in df.columns:
            bad = (df['dynamic_range'] < self.rules['dynamic_range']['min_db']) & \
                  (df['dynamic_range'].notna())
            scores[bad] -= self.rules['dynamic_range']['penalty']
            for idx in df[bad].index:
                issues[idx].append(self.rules['dynamic_range']['reason'])

        return np.clip(scores, 0, 1), issues


def compute_quality_scores_audio(df):
    """Compute quality scores using AudioQualityScorer."""
    scorer = AudioQualityScorer()
    scores, issues = scorer.score(df)
    df['quality_score'] = scores
    df['quality_issues'] = [issues[i] for i in range(len(df))]
    return df


# 🆕 ADD THIS VISUALIZATION FUNCTION
# ────────────────────────────────────────────────────────────────────────

def plot_quality_analysis_simple(df):
    """Simpler version of quality analysis for your data format."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # 1. Quality distribution
    axes[0, 0].hist(df['quality_score'], bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
    axes[0, 0].axvline(df['quality_score'].mean(), color='red', linestyle='--', linewidth=2,
                       label=f'Mean: {df["quality_score"].mean():.3f}')
    axes[0, 0].set_xlabel('Quality Score')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Quality Score Distribution')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Speech rate vs quality
    axes[0, 1].scatter(df['speech_rate_wps'], df['quality_score'],
                      c=df['quality_score'], cmap='RdYlGn', alpha=0.6, s=30)
    axes[0, 1].axhline(0.7, color='red', linestyle='--', alpha=0.5)
    axes[0, 1].set_xlabel('Speech Rate (words/sec)')
    axes[0, 1].set_ylabel('Quality Score')
    axes[0, 1].set_title('Speech Rate vs Quality')
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Duration distribution
    axes[0, 2].hist(df['duration_ms']/1000, bins=40, color='#A23B72', alpha=0.7, edgecolor='black')
    axes[0, 2].set_xlabel('Duration (seconds)')
    axes[0, 2].set_ylabel('Count')
    axes[0, 2].set_title('Segment Duration')
    axes[0, 2].grid(True, alpha=0.3, axis='y')

    # 4. Word count
    axes[1, 0].hist(df['n_words'], bins=40, color='#F18F01', alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Number of Words')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Words per Segment')
    axes[1, 0].grid(True, alpha=0.3, axis='y')

    # 5. Speech activity (if available)
    if 'speech_activity_ratio' in df.columns:
        axes[1, 1].hist(df['speech_activity_ratio'].dropna(), bins=40, color='#06A77D', alpha=0.7, edgecolor='black')
        axes[1, 1].axvline(0.4, color='red', linestyle='--', linewidth=2, label='Threshold')
        axes[1, 1].set_xlabel('Speech Activity Ratio')
        axes[1, 1].set_ylabel('Count')
        axes[1, 1].set_title('Speech Activity Distribution')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3, axis='y')

    # 6. Quality by speaker
    if 'speaker' in df.columns:
        speaker_quality = df.groupby('speaker')['quality_score'].mean().sort_values(ascending=False).head(10)
        axes[1, 2].barh(range(len(speaker_quality)), speaker_quality.values, color='#2E86AB', alpha=0.7, edgecolor='black')
        axes[1, 2].set_yticks(range(len(speaker_quality)))
        axes[1, 2].set_yticklabels(speaker_quality.index, fontsize=9)
        axes[1, 2].set_xlabel('Average Quality Score')
        axes[1, 2].set_title('Quality by Speaker (Top 10)')
        axes[1, 2].grid(True, alpha=0.3, axis='x')

    plt.tight_layout()
    return fig


# 🆕 ADD THIS FILTERING AND REPORTING FUNCTION
# ────────────────────────────────────────────────────────────────────────

def filter_and_report_quality(df, threshold=0.7):
    """Filter segments by quality and print detailed report."""

    print(f"\n{'='*70}")
    print("DATA QUALITY ANALYSIS & FILTERING")
    print(f"{'='*70}\n")

    print(f"📊 OVERALL STATISTICS")
    print(f"   Total segments analyzed: {len(df):,}")
    print(f"   Quality score range: [{df['quality_score'].min():.3f}, {df['quality_score'].max():.3f}]")
    print(f"   Mean quality score: {df['quality_score'].mean():.3f}")
    print(f"   Median quality score: {df['quality_score'].median():.3f}\n")

    # Filter
    high_quality = df[df['quality_score'] >= threshold].copy()
    removed = df[df['quality_score'] < threshold].copy()

    print(f"🎯 FILTERING RESULTS (threshold: {threshold})")
    print(f"   ✓ KEPT:    {len(high_quality):6d} ({len(high_quality)/len(df)*100:5.1f}%)  HIGH-QUALITY")
    print(f"   ✗ REMOVED: {len(removed):6d} ({len(removed)/len(df)*100:5.1f}%)  LOW-QUALITY\n")

    # Issue analysis
    print(f" WHY SEGMENTS WERE REMOVED")
    issue_counts = {}
    for issues_list in removed['quality_issues']:
        for issue in issues_list:
            issue_counts[issue] = issue_counts.get(issue, 0) + 1

    if issue_counts:
        for issue, count in sorted(issue_counts.items(), key=lambda x: -x[1]):
            pct = count / len(removed) * 100 if len(removed) > 0 else 0
            print(f"   • {issue:50s}: {count:5d} ({pct:5.1f}%)")
    else:
        print(f"   (no specific issues to report)")

    # Stats on high-quality segments
    if len(high_quality) > 0:
        print(f"\n HIGH-QUALITY SEGMENTS STATISTICS")
        print(f"   Avg duration: {high_quality['duration_ms'].mean()/1000:.2f}s")
        print(f"   Avg words: {high_quality['n_words'].mean():.1f}")
        print(f"   Avg speech rate: {high_quality['speech_rate_wps'].mean():.2f} words/sec")
        if 'speech_activity_ratio' in high_quality.columns:
            print(f"   Avg speech activity: {high_quality['speech_activity_ratio'].mean():.2%}")

    print(f"\n{'='*70}\n")

    return high_quality, removed

In [17]:
# ════════════════════════════════════════════════════════════════════════
# 🆕 ZONE 5: DATA QUALITY ANALYSIS (NEW STEP!)
# ════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("ZONE 5: DATA QUALITY ANALYSIS (NEW!)")
print("="*70)

# Step 5b: Extract features directly from audio_segments
print("\n[5b] Extracting features...")
df = extract_audio_segment_features(audio_segments)

# Step 5c: Compute quality scores
print("\n[5c] Computing quality scores...")
df = compute_quality_scores_audio(df)

# Step 5d: Visualize
print("\n[5d] Generating visualizations...")
fig = plot_quality_analysis_simple(df)

# Save visualization
viz_path = CONFIG["output_dir"] / "quality_analysis_dashboard.png"
fig.savefig(viz_path, dpi=150, bbox_inches='tight')
print(f"   ✓ Saved visualization to {viz_path}")
plt.close(fig)

# Step 5e: Analyze and filter
print("\n[5e] Filtering data...")
high_quality_df, removed_df = filter_and_report_quality(
    df,
    threshold=CONFIG["quality_threshold"]
)

# Step 5f: Export analysis results (assuming you want to save filtered data)
print("\n[5f] Exporting analysis results...")
analysis_dir = CONFIG["output_dir"] / "quality_analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
if CONFIG["save_filtered_segments"]:
    high_quality_df.to_csv(analysis_dir / "high_quality_segments.csv", index=False)
    removed_df.to_csv(analysis_dir / "removed_segments.csv", index=False)
    print(f"   ✓ Saved filtered segments to {analysis_dir}")

# Prepare audio_segments for Whisper evaluation after quality filtering
# Create a set of segment_ids for high quality segments
high_quality_segment_ids = set(high_quality_df['segment_id'])

# Filter original audio_segments based on high quality segment_ids
filtered_audio_segments = [
    seg for seg in audio_segments if seg['segment_id'] in high_quality_segment_ids
]

# Update audio_segments to only contain high-quality segments for subsequent steps
audio_segments = filtered_audio_segments


ZONE 5: DATA QUALITY ANALYSIS (NEW!)

[5b] Extracting features...

EXTRACTING FEATURES FROM AUDIO SEGMENTS



Processing segments:   0%|          | 0/1000 [00:00<?, ?it/s]

  ⚠️  Error loading /content/drive/MyDrive/asr/output/whisper_children_dataset/audio_segments/KAT/01-3a_KAT_00294.wav: can't extend empty axis 0 using modes other than 'constant' or 'empty'
  ⚠️  Error loading /content/drive/MyDrive/asr/output/whisper_children_dataset/audio_segments/KAT/01-3a_KAT_00295.wav: can't extend empty axis 0 using modes other than 'constant' or 'empty'
  ⚠️  Error loading /content/drive/MyDrive/asr/output/whisper_children_dataset/audio_segments/WIL/01-3a_WIL_00296.wav: can't extend empty axis 0 using modes other than 'constant' or 'empty'
  ⚠️  Error loading /content/drive/MyDrive/asr/output/whisper_children_dataset/audio_segments/WIL/01-3a_WIL_00297.wav: can't extend empty axis 0 using modes other than 'constant' or 'empty'
  ⚠️  Error loading /content/drive/MyDrive/asr/output/whisper_children_dataset/audio_segments/MAS/01-3a_MAS_00298.wav: can't extend empty axis 0 using modes other than 'constant' or 'empty'
  ⚠️  Error loading /content/drive/MyDrive/asr/out

## Zone 6: Whisper Baseline Evaluation Compute of the WER before fine-tuning

### Whisper Evaluation Class

In [21]:
@dataclass
class TranscriptionResult:
    segment_id: str
    speaker: str
    file_name: str
    audio_path: str
    ground_truth: str
    whisper_prediction: str
    duration_ms: int
    wer: float
    confidence: float = 0.0


class WhisperEvaluator:
    """Evaluate Whisper baseline on children voices"""

    def __init__(self, model_name: str = "base"):
        print(f"Loading Whisper '{model_name}'...")
        self.model = whisper.load_model(model_name)
        print("Loaded\n")

    def transcribe(self, audio_path: Path) -> Dict:
        if not audio_path.exists():
            return {"text": "", "confidence": 0.0}
        try:
            result = self.model.transcribe(str(audio_path), language="fr", verbose=False)
            return {"text": result["text"].strip(), "confidence": result.get("confidence", 0.0)}
        except:
            return {"text": "", "confidence": 0.0}

    @staticmethod
    def calculate_wer(ground_truth: str, prediction: str) -> float:
        if not ground_truth.strip():
            return 0.0 if not prediction.strip() else 1.0
        return compute_wer(ground_truth, prediction)

    def evaluate_children(self, audio_segments: List[Dict], speakers_info: Dict,
                         sample_size: int = None) -> List[TranscriptionResult]:
        """Evaluate only children speakers"""

        print("="*70)
        print("STEP 4: WHISPER BASELINE EVALUATION (CHILDREN ONLY)")
        print("="*70)

        # Filter children only
        children_segments = [s for s in audio_segments
                            if speakers_info.get(s["speaker"]) in CHILD_ROLES]

        if sample_size:
            children_segments = children_segments[:sample_size]

        print(f"\nEvaluating {len(children_segments)} children segments\n")

        results = []
        for i, seg in enumerate(children_segments):
            audio_path = Path(seg["audio_path"])
            transcription = self.transcribe(audio_path)
            wer = self.calculate_wer(seg["text"], transcription["text"])

            results.append(TranscriptionResult(
                segment_id=seg["segment_id"],
                speaker=seg["speaker"],
                file_name=seg["file_name"],
                audio_path=seg["audio_path"],
                ground_truth=seg["text"],
                whisper_prediction=transcription["text"],
                duration_ms=seg["duration_ms"],
                wer=wer,
                confidence=transcription["confidence"]
            ))

            if (i + 1) % 50 == 0:
                avg_wer = sum(r.wer for r in results) / len(results)
                print(f"  ✓ {i + 1}/{len(children_segments)} | Avg WER: {avg_wer:.3f}")

        return results


### WER Report

In [22]:

def print_wer_report(results: List[TranscriptionResult], speakers_info: Dict):
    """Print WER statistics"""

    wers = [r.wer for r in results]

    print("\n" + "="*70)
    print("STEP 5: WER STATISTICS (CHILDREN ONLY)")
    print("="*70)

    print(f"\nGlobal:")
    print(f"   Total segments:  {len(results)}")
    print(f"   Avg WER:         {sum(wers) / len(wers):.3f}")
    print(f"   Min WER:         {min(wers):.3f}")
    print(f"   Max WER:         {max(wers):.3f}")
    print(f"   Median WER:      {sorted(wers)[len(wers)//2]:.3f}")

    print(f"\n   Distribution:")
    for low, high in [(0.0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 1.0)]:
        count = sum(1 for w in wers if low <= w < high)
        pct = (count / len(wers)) * 100
        print(f"      {low:.1f}-{high:.1f}: {count:4d} ({pct:5.1f}%)")

    # By speaker
    by_speaker = {}
    for r in results:
        by_speaker.setdefault(r.speaker, []).append(r.wer)

    print(f"\n👥 By speaker ({len(by_speaker)}):")
    for speaker in sorted(by_speaker.keys()):
        wers_sp = by_speaker[speaker]
        avg = sum(wers_sp) / len(wers_sp)
        print(f"      {speaker:15} {len(wers_sp):4d} segments | WER: {avg:.3f}")

    print("\n" + "="*70 + "\n")


In [23]:
 # ════════════════════════════════════════════════════════════════════════
    # ZONE 6: Whisper Evaluation (ONLY on high-quality segments!)
    # ════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("ZONE 6: WHISPER EVALUATION (HIGH-QUALITY SEGMENTS ONLY)")
print("="*70)

    # Filter audio_segments to only include high-quality ones
high_quality_ids = set(high_quality_df['speaker'].unique())
filtered_audio_segments = [
    seg for seg in audio_segments
    if seg['speaker'] in high_quality_ids  # Or use segment_id if available
]

print(f"\n   📊 Using {len(filtered_audio_segments)}/{len(audio_segments)} segments for Whisper")
print(f"      (Filtered: {len(audio_segments) - len(filtered_audio_segments)} low-quality segments removed)")

evaluator = WhisperEvaluator(CONFIG["whisper_model"])
results = evaluator.evaluate_children(
    filtered_audio_segments,  # HIGH-QUALITY ONLY!
    speakers_info,
    CONFIG["sample_size"]
)

print_wer_report(results, speakers_info)



ZONE 6: WHISPER EVALUATION (HIGH-QUALITY SEGMENTS ONLY)

   📊 Using 79/79 segments for Whisper
      (Filtered: 0 low-quality segments removed)
Loading Whisper 'base'...
Loaded

STEP 4: WHISPER BASELINE EVALUATION (CHILDREN ONLY)

Evaluating 25 children segments



100%|██████████| 1321/1321 [00:08<00:00, 149.89frames/s]
0frames [00:00, ?frames/s]
0frames [00:00, ?frames/s]
100%|██████████| 1588/1588 [00:05<00:00, 316.50frames/s]
0frames [00:00, ?frames/s]
0frames [00:00, ?frames/s]
100%|██████████| 554/554 [00:01<00:00, 442.52frames/s]


STEP 5: WER STATISTICS (CHILDREN ONLY)

Global:
   Total segments:  25
   Avg WER:         2.237
   Min WER:         0.917
   Max WER:         7.333
   Median WER:      1.500

   Distribution:
      0.0-0.1:    0 (  0.0%)
      0.1-0.3:    0 (  0.0%)
      0.3-0.5:    0 (  0.0%)
      0.5-1.0:    1 (  4.0%)

👥 By speaker (10):
      CAR                1 segments | WER: 1.000
      CHI                1 segments | WER: 3.000
      KLO                3 segments | WER: 1.515
      LUS                8 segments | WER: 2.496
      MAI                4 segments | WER: 3.458
      MAT                3 segments | WER: 1.756
      NIN                2 segments | WER: 1.256
      RIT                1 segments | WER: 3.800
      SAR                1 segments | WER: 1.000
      WIL                1 segments | WER: 1.000





## Zone 7: Create Training Dataset

In [24]:
# When i am done i have to analyze the datas before creation of the training set


class DatasetBuilder:
    """Create train/test splits for fine-tuning"""

    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def create_dataset(self, results: List[TranscriptionResult], train_ratio: float = 0.8):
        """Create JSONL + metadata files"""

        print("="*70)
        print("STEP 6: CREATE TRAINING DATASET")
        print("="*70)

        split_idx = int(len(results) * train_ratio)
        train = results[:split_idx]
        test = results[split_idx:]

        print(f"\nDataset split:")
        print(f"   Total:   {len(results)} segments")
        print(f"   Train:   {len(train)} segments ({train_ratio*100:.0f}%)")
        print(f"   Test:    {len(test)} segments ({(1-train_ratio)*100:.0f}%)")

        # Save JSONL (for fine-tuning)
        self._save_jsonl(train, self.output_dir / "train.jsonl")
        self._save_jsonl(test, self.output_dir / "eval.jsonl")

        # Save metadata JSON (for analysis)
        self._save_metadata(train, self.output_dir / "train_metadata.json")
        self._save_metadata(test, self.output_dir / "eval_metadata.json")

        print(f"\nDataset created in {self.output_dir}\n")

    def _save_jsonl(self, results: List[TranscriptionResult], output_file: Path):
        """Save as JSONL for Whisper"""
        with open(output_file, "w", encoding="utf-8") as f:
            for r in results:
                entry = {"audio": r.audio_path, "text": r.ground_truth, "language": "fr"}
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"   ✓ {output_file.name} ({len(results)} segments)")

    def _save_metadata(self, results: List[TranscriptionResult], output_file: Path):
        """Save complete metadata"""
        data = [asdict(r) for r in results]
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        print(f"   ✓ {output_file.name}")

In [25]:

    # ════════════════════════════════════════════════════════════════════════
    # ZONE 7: Create Training Dataset
    # ════════════════════════════════════════════════════════════════════════

print("\n" + "="*70)
print("ZONE 7: CREATE TRAINING DATASET")
print("="*70)

builder = DatasetBuilder(CONFIG["output_dir"] / "training_dataset")
builder.create_dataset(results, CONFIG["train_ratio"])



ZONE 7: CREATE TRAINING DATASET
STEP 6: CREATE TRAINING DATASET

Dataset split:
   Total:   25 segments
   Train:   20 segments (80%)
   Test:    5 segments (20%)
   ✓ train.jsonl (20 segments)
   ✓ eval.jsonl (5 segments)
   ✓ train_metadata.json
   ✓ eval_metadata.json

Dataset created in /content/drive/MyDrive/asr/output/whisper_children_dataset/training_dataset



## MAIN PIPELINE

In [ ]:
# ============================================================================
# INTEGRATED PIPELINE WITH QUALITY ANALYSIS
# ============================================================================

def run_pipeline_with_analysis():
    """Run complete pipeline with quality analysis BEFORE Whisper evaluation"""

    print("\n" + "="*70)
    print("COMPLETE WHISPER CHILDREN DATASET PIPELINE (WITH ANALYSIS)")
    print("="*70)

    # ZONE 1: Matching
    print("\n" + "="*70)
    print("ZONE 1: FILE MATCHING")
    print("="*70)
    match_result = find_matching_files(CONFIG["cha_dir"], CONFIG["audio_dir"], CONFIG["audio_extensions"])
    print_matching_report(match_result)

    if not match_result["matched"]:
        print("❌ No matched pairs found!")
        return

    # ZONE 2: Extract segments from .cha files
    print("\n" + "="*70)
    print("ZONE 2: EXTRACT SEGMENTS FROM .CHA")
    print("="*70)
    segments = extract_segments_from_matched(match_result["matched"])

    # ZONE 3: Segment audio files
    print("\n" + "="*70)
    print("ZONE 3: CREATE AUDIO SEGMENTS")
    print("="*70)
    segmenter = AudioSegmenter(CONFIG["output_dir"] / "audio_segments")
    segment_all_result = segmenter.segment_all(segments, match_result["matched"], max_segments=None)
    audio_segments = segment_all_result["extracted"]

    # ZONE 4: Get speaker info
    print("\n" + "="*70)
    print("ZONE 4: EXTRACT SPEAKER ROLES")
    print("="*70)
    speakers_info = extract_all_speakers_info(match_result["matched"])

    # ════════════════════════════════════════════════════════════════════════
    # 🆕 ZONE 5: DATA QUALITY ANALYSIS (NEW STEP!)
    # ════════════════════════════════════════════════════════════════════════

    print("\n" + "="*70)
    print("ZONE 5: DATA QUALITY ANALYSIS (NEW!)")
    print("="*70)

    # Step 5a: Convert audio_segments back to WorSegment format for analysis
    print("\n[5a] Converting segments to analysis format...")
    analysis_segments = []
    for audio_seg in audio_segments:
        # Create WorSegment for analysis
        wor_seg = WorSegment(
            speaker=audio_seg["speaker"],
            text=audio_seg["text"],
            words=[(w, s, e) for w, s, e in zip(
                audio_seg["text"].split(),
                [0] * len(audio_seg["text"].split()),  # Placeholder timestamps
                [0] * len(audio_seg["text"].split())
            )],  # Note: We use text-based words, actual timing from audio_path
            audio_path=Path(audio_seg["audio_path"]),
            cha_path=Path("")  # Not needed for analysis
        )
        analysis_segments.append(wor_seg)

    # Step 5b: Extract features
    print("\n[5b] Extracting features...")
    df = analyze_dataset(analysis_segments, compute_audio_features_flag=True)
    # Set compute_audio_features_flag=False if you don't have access to audio files
    # or want to speed things up

    # Step 5c: Compute quality scores
    print("\n[5c] Computing quality scores...")
    df = compute_quality_scores(df)

    # Step 5d: Visualize
    print("\n[5d] Generating visualizations...")
    fig = plot_comprehensive_analysis(df)

    # Save visualization
    viz_path = CONFIG["output_dir"] / "quality_analysis_dashboard.png"
    fig.savefig(viz_path, dpi=150, bbox_inches='tight')
    print(f"   ✓ Saved visualization to {viz_path}")
    plt.close(fig)

    # Step 5e: Analyze and filter
    print("\n[5e] Filtering data...")
    high_quality_df, removed_df, analysis = analyze_and_filter(
        df,
        thresholds={'primary': 0.7},  # Adjust this threshold!
        verbose=True
    )

    # Step 5f: Detailed problem analysis
    print_detailed_issues(df, top_n=30)

    # Step 5g: Export analysis results
    print("\n[5g] Exporting analysis results...")
    analysis_dir = CONFIG["output_dir"] / "quality_analysis"
    export_filtered_data(high_quality_df, removed_df, analysis_dir)

    # ════════════════════════════════════════════════════════════════════════
    # ZONE 6: Whisper Evaluation (ONLY on high-quality segments!)
    # ════════════════════════════════════════════════════════════════════════

    print("\n" + "="*70)
    print("ZONE 6: WHISPER EVALUATION (HIGH-QUALITY SEGMENTS ONLY)")
    print("="*70)

    # Filter audio_segments to only include high-quality ones
    high_quality_ids = set(high_quality_df['speaker'].unique())
    filtered_audio_segments = [
        seg for seg in audio_segments
        if seg['speaker'] in high_quality_ids  # Or use segment_id if available
    ]

    print(f"\n   📊 Using {len(filtered_audio_segments)}/{len(audio_segments)} segments for Whisper")
    print(f"      (Filtered: {len(audio_segments) - len(filtered_audio_segments)} low-quality segments removed)")

    evaluator = WhisperEvaluator(CONFIG["whisper_model"])
    results = evaluator.evaluate_children(
        filtered_audio_segments,  # HIGH-QUALITY ONLY!
        speakers_info,
        CONFIG["sample_size"]
    )

    print_wer_report(results, speakers_info)

    # ════════════════════════════════════════════════════════════════════════
    # ZONE 7: Create Training Dataset
    # ════════════════════════════════════════════════════════════════════════

    print("\n" + "="*70)
    print("ZONE 7: CREATE TRAINING DATASET")
    print("="*70)

    builder = DatasetBuilder(CONFIG["output_dir"] / "training_dataset")
    builder.create_dataset(results, CONFIG["train_ratio"])

    # ════════════════════════════════════════════════════════════════════════
    # FINAL SUMMARY
    # ════════════════════════════════════════════════════════════════════════

    print("\n" + "="*70)
    print("✅ PIPELINE COMPLETE")
    print("="*70)

    print(f"\n📊 SUMMARY:")
    print(f"   Original segments:        {len(audio_segments):,}")
    print(f"   High-quality segments:    {len(high_quality_df):,} ({len(high_quality_df)/len(df)*100:.1f}%)")
    print(f"   Whisper-evaluated:        {len(results):,}")
    print(f"   Training set:             {len(results) * CONFIG['train_ratio']:.0f}")
    print(f"   Evaluation set:           {len(results) * (1-CONFIG['train_ratio']):.0f}")

    print(f"\n📁 OUTPUT DIRECTORIES:")
    print(f"   Quality analysis:         {analysis_dir}/")
    print(f"   Training dataset:         {CONFIG['output_dir']}/training_dataset/")
    print(f"   Audio segments:           {CONFIG['output_dir']}/audio_segments/")

    print(f"\n📋 KEY FILES:")
    print(f"   high_quality_segments.csv .... segments that passed quality filter")
    print(f"   removed_segments.csv ......... segments that were filtered out")
    print(f"   quality_analysis_dashboard.png  visualization of data quality")
    print(f"   train.jsonl .................. ready for fine-tuning")
    print(f"   eval.jsonl ................... ready for evaluation")

    print(f"\n{'='*70}\n")


if __name__ == "__main__":
    run_pipeline_with_analysis()


COMPLETE WHISPER CHILDREN DATASET PIPELINE

STEP 1: FILE MATCHING (by relative path)

 Found:
   Total .cha files:       247
   Total audio files:      245
   Matched pairs:        245
   .cha missing audio:  2
   Audio orphans:       0

   Missing audio for:
      - 03-13c.cha
      - 05-23a.cha


STEP 2: EXTRACT .CHA SEGMENTS
  ✓ 50/245 files
  ✓ 100/245 files
  ✓ 150/245 files
  ✓ 200/245 files

Extracted 29272 segments with timestamps

STEP 3: SEGMENT AUDIO FILES
  ✓ 200/29272 segments
  ✓ 400/29272 segments
  ✓ 600/29272 segments
  ✓ 800/29272 segments
  ✓ 1000/29272 segments
  ✓ 1200/29272 segments
  ✓ 1400/29272 segments
  ✓ 1600/29272 segments
  ✓ 1800/29272 segments
  ✓ 2000/29272 segments
  ✓ 2200/29272 segments
  ✓ 2400/29272 segments
  ✓ 2600/29272 segments
  ✓ 2800/29272 segments
  ✓ 3000/29272 segments
  ✓ 3200/29272 segments
  ✓ 3400/29272 segments
  ✓ 3600/29272 segments
  ✓ 3800/29272 segments
  ✓ 4000/29272 segments
  ✓ 4200/29272 segments
  ✓ 4400/29272 segments
  ✓